In [ ]:
import shutil
import os
import tempfile
import math
from tqdm import tqdm

import torch.nn.functional as F
from datasets import Dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

In [ ]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ["WANDB_DISABLED"] = "true"

EVALUATION = False
ROOT_PATH = "/personal/chameleon" if not EVALUATION else "/bohr/train-7ul9/v2"

# Data

In [ ]:
hint_description = Dataset.load_from_disk(ROOT_PATH + "/dataset/hint_descriptions")
hint_description = {
    x['ID']: {'description': x['Description'], 'icons': x['image']}
    for x in hint_description
}

hint_description[7]['description']

'Flora\nPlant\nNature'

In [ ]:
validation_data = Dataset.load_from_disk(ROOT_PATH + "/dataset/takehome_validation")

validation_data.to_pandas().head()

,hints,options,label
0,"[6, 61, 63, 115, 33]","[sunflower, credit card, dinosaur, key, sundia...",seal
1,"[24, 91, 114, 110, 109, 108, 107, 106, 105, 78]","[bread, calculator, credit card, surfboard, zi...",billiards
2,"[4, 34, 60]","[alumunium foil, train tracks, octopus, mounta...",firefighters
3,"[6, 20, 58, 105, 52]","[cake, bathtub, peanuts, jellyfish, zebra, cro...",bat
4,"[18, 54, 76, 57]","[bat, socks, garage, postal worker, daisy, rai...",eclipse


# Implement keyword guesser

In [ ]:
model_path = "Qwen/Qwen3-Embedding-0.6B" if not EVALUATION else "/bohr/pretrained-models-ewgg/v3/qwen3-0.6B"

model = SentenceTransformer(model_path)

In [ ]:
def hints_to_sentence_v1(hints: list[int]) -> str:
    descriptions = [hint_description[hint]['description'].replace('\n', ', ') for hint in hints]

    sentence = f"You are playing a word guessing game.\n The target concept: {descriptions[0]}.\n"
    sentence += f"Based on the following clues, try to infer the secret word: "
    for i, desc in enumerate(descriptions[1:]):
        sentence += f"{i}. {desc} <SEP> "

    return sentence

def hints_to_sentence(hints: list[int]) -> str:
    descriptions = [hint_description[hint]['description'].replace('\n', ', ') for hint in hints]

    sentence = f"Target concept: {descriptions[0]}"
    sentence += f" <SEP> Context clues: {' -> '.join(descriptions[1:])}"

    # add semantic relationships
    sentence += f" <SEP> Full sequence: {' then '.join(descriptions)}"

    return sentence

In [ ]:
def create_multiple_queries(hints):
    queries = []

    queries.append(hints_to_sentence(hints))
    queries.append(hints_to_sentence_v1(hints))

    return queries

In [ ]:
normalize = lambda v: v / np.sqrt(np.sum(v**2))

def guess_words(
    hints: list[int],
    choices: list[str],
) -> list[str]:
    # 0. generate prompts
    queries = create_multiple_queries(hints)

    # 1. get the embeddings
    query_embeddings = normalize(model.encode(queries))
    choice_embeddings = normalize(model.encode(choices))

    # 2. compute  cosine sim
    all_similarities = []
    for query_emb in query_embeddings:
        similarities = cosine_similarity([query_emb], choice_embeddings)[0]
        all_similarities.append(similarities)

    # 3. ensemble: weighted average
    weights = [0.9, 0.1]
    final_similarities = np.average(all_similarities, axis=0, weights=weights)

    # 3. take top 10 candidates
    top_indices = np.argsort(final_similarities)[::-1][:10]

    return [choices[idx] for idx in top_indices]

In [ ]:
len(guess_words(validation_data[0]['hints'], validation_data[0]['options'])) == 10

True

In [ ]:
hints_to_sentence(validation_data[0]['hints'])

'Target concept: Fauna, Animal <SEP> Context clues: Water, Liquid, Aquatic -> Earth, Ground -> Grey -> Fast, Race <SEP> Full sequence: Fauna, Animal then Water, Liquid, Aquatic then Earth, Ground then Grey then Fast, Race'

# Finetune

In [ ]:
def choice_to_doc(choice:str)->str:
  return f"Our target word: {choice}"

In [ ]:
train_examples = []
for val in validation_data:
  train_examples.append(InputExample(texts=[hints_to_sentence(val['hints']), choice_to_doc(val['label'])], label=1))

# Create DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)

# Define loss function
train_loss = losses.CosineSimilarityLoss(model)

# model.fit(
#     train_objectives=[(train_dataloader, train_loss)],
#     epochs=1,
#     warmup_steps=5,
#     output_path='./model',
#     optimizer_params={'lr': 1e-6},
#     weight_decay=0.01,
#     save_best_model=True,
#     show_progress_bar=True
# )

# Evaluation

In [ ]:
ranks = []

def score(guesses: list[str], gold: str):

    # Normalize to lowercase
    guesses = [g.lower() for g in guesses[:10]]
    gold = gold.lower()

    result = {
        "hits@10": 0.0,
        "ndcg@10": 0.0,
        "total_score": 0.0
    }

    if gold in guesses:
        rank = guesses.index(gold)
        ranks.append(rank)
        result["hits@10"] = 1.0
        result["ndcg@10"] = 1.0 / math.log2(rank + 2)  # rank + 2 because index is 0-based
    else:
        result["hits@10"] = 0.0
        result["ndcg@10"] = 0.0

    result["total_score"] = 0.9 * result["hits@10"] + 0.1 * result["ndcg@10"]
    return result

# score(['cat', 'dog', 'tree', 'flower', 'rock', 'water', 'fried rice', 'airplane', 'cactus', 'tiger'], gold='cactus')

In [ ]:
guesses = []
total_scores, hits = 0.0, 0.0
for example in tqdm(validation_data):
    guesses.append(guess_words(example['hints'], example['options']))
    s = score(guesses[-1], example['label'])

    total_scores += s['total_score']
    hits += s['hits@10']

print(f"Average validation score: {total_scores / len(validation_data)}")
print(f"Average hits@10: {hits / len(validation_data)}")

100%|██████████| 20/20 [00:02<00:00,  7.33it/s]

Average validation score: 0.6311387116319666
Average hits@10: 0.65


In [ ]:
ranks

[3, 1, 2, 0, 0, 0, 0, 9, 7, 1, 3, 0, 0]

# Submission

In [ ]:
model_code = """
from sentence_transformers import SentenceTransformer, CrossEncoder
from datasets import Dataset
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

EVALUATION = True
ROOT_PATH = "/personal/chameleon" if not EVALUATION else "/bohr/train-7ul9/v2"
hint_description = Dataset.load_from_disk(ROOT_PATH + "/dataset/hint_descriptions")
hint_description = {
    x['ID']: {'description': x['Description'], 'icons': x['image']}
    for x in hint_description
}

model_path = "Qwen/Qwen3-Embedding-0.6B" if not EVALUATION else "/bohr/pretrained-models-ewgg/v3/qwen3-0.6B"

model = SentenceTransformer(model_path)

def hints_to_sentence_v1(hints: list[int]) -> str:
    descriptions = [hint_description[hint]['description'].replace('\\n', ', ') for hint in hints]

    sentence = f"You are playing a word guessing game.\\n The target concept: {descriptions[0]}.\\n"
    sentence += f"Based on the following clues, try to infer the secret word: "
    for i, desc in enumerate(descriptions[1:]):
        sentence += f"{i}. {desc} <SEP> "

    return sentence

def hints_to_sentence(hints: list[int]) -> str:
    descriptions = [hint_description[hint]['description'].replace('\\n', ', ') for hint in hints]

    sentence = f"Target concept: {descriptions[0]}"
    sentence += f" <SEP> Context clues: {' -> '.join(descriptions[1:])}"

    # add semantic relationships
    sentence += f" <SEP> Full sequence: {' then '.join(descriptions)}"

    return sentence

def create_multiple_queries(hints):
    queries = []

    queries.append(hints_to_sentence(hints))
    queries.append(hints_to_sentence_v1(hints))

    return queries

normalize = lambda v: v / np.sqrt(np.sum(v**2))

def guess_words(
    hints: list[int],
    choices: list[str],
) -> list[str]:
    # 0. generate prompts
    queries = create_multiple_queries(hints)

    # 1. get the embeddings
    query_embeddings = normalize(model.encode(queries))
    choice_embeddings = normalize(model.encode(choices))

    # 2. compute  cosine sim
    all_similarities = []
    for query_emb in query_embeddings:
        similarities = cosine_similarity([query_emb], choice_embeddings)[0]
        all_similarities.append(similarities)

    # 3. ensemble: weighted average
    weights = [0.6, 0.4]
    final_similarities = np.average(all_similarities, axis=0, weights=weights)

    # 3. take top 10 candidates
    top_indices = np.argsort(final_similarities)[::-1][:10]

    return [choices[idx] for idx in top_indices]
"""

if EVALUATION:
  with open("submission_model.py", "w") as f:
    f.write(model_code)

  print("Inference code written to submission_model.py")

In [ ]:
if EVALUATION:
    # Create a temporary directory with your desired structure
    with tempfile.TemporaryDirectory() as temp_dir:
        # Copy files to temp directory
        shutil.copy('submission_model.py', temp_dir)
        shutil.copytree('/model', os.path.join(temp_dir, 'model'))

        # Create the zip
        shutil.make_archive('submission', 'zip', temp_dir)